# Notebook 4 — Comprensione de immagini

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lezione 4:** Understanding Images (crop hints, rilevamento de volti, propiedades de la immagine)

In [ ]:
import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision
client = vision.ImageAnnotatorClient()
print("Cliente listo.")


## 1. Sugerencias de recorte (`crop_hints`)

El metodo `crop_hints` sugiere la región más "saliente" (la que concentra el contenuto visual más relevante) de una immagine, come semilla para recortarla según una relación de aspecto que tú indiques. Es útil para adaptar immagini grandes a distintos formatos (miniaturas, banners, vistas móviles) sin perder el punto de interés principal.

**Parámetros que puedes ajustar:**

- **`image`**: JPEG o PNG, hasta 32 megabytes
- **`max_results`**: número máximo de sugerencias a devolver, de uno a diez
- **`aspect_ratios`**: lista de relaciones ancho/alto — por esempio, `[1.0, 1.333]` para 1:1 y 4:3
- **`crop_hints_params.confidence_threshold`**: descarta sugerencias por debajo de un umbral de confidenza

In [ ]:
def detect_crop_hints(image_path, aspect_ratios):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    crop_hints_params = vision.CropHintsParams(aspect_ratios=aspect_ratios)
    image_context = vision.ImageContext(crop_hints_params=crop_hints_params)

    response = client.crop_hints(image=image, image_context=image_context)

    for hint in response.crop_hints_annotation.crop_hints:
        print("Cuadro delimitador:", hint.bounding_poly)
        print("Confianza:", hint.confidence)
        print("Fracción de importancia:", hint.importance_fraction)

    return response.crop_hints_annotation.crop_hints


# detect_crop_hints("paisaje.jpg", aspect_ratios=[16 / 9, 4 / 3, 1.0])


## Ejercicio 1 — Recorte automático real

Completa `recortar_immagine` para que, usando el `bounding_poly` de la primera sugerencia de `crop_hints`, calcule las coordenadas mínimas y máximas y aplique `PIL.Image.crop()` para generar y guardar el recorte final.

Pista: `Image.crop()` espera una tupla `(izquierda, arriba, derecha, abajo)`.

In [ ]:
from PIL import Image

def recortar_imagen(image_path, aspect_ratios, output_path):
    hints = detect_crop_hints(image_path, aspect_ratios)
    vertices = hints[0].bounding_poly.vertices

    # TODO 1: calcula min_x, max_x, min_y, max_y a partir de los vértices
    min_x = ...
    max_x = ...
    min_y = ...
    max_y = ...

    # TODO 2: abre la imagen original con PIL, recórtala y guárdala en output_path
    imagen = ...
    recorte = ...
    recorte.save(output_path)

    return output_path


# recortar_imagen("paisaje.jpg", [1.0], "paisaje_recortado.jpg")


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
from PIL import Image

def recortar_immagine(image_path, aspect_ratios, output_path):
    hints = detect_crop_hints(image_path, aspect_ratios)
    vertices = hints[0].bounding_poly.vertices

    min_x = min(v.x for v in vertices)
    max_x = max(v.x for v in vertices)
    min_y = min(v.y for v in vertices)
    max_y = max(v.y for v in vertices)

    immagine = Image.open(image_path)
    recorte = immagine.crop((min_x, min_y, max_x, max_y))
    recorte.save(output_path)

    return output_path


recortar_immagine("paisaje.jpg", [1.0], "paisaje_recortado.jpg")
```

</details>

## 2. Detección de volti (`face_detection`)

`face_detection` localiza uno o varios volti en una immagine y, por cada uno, te da su riquadro delimitador, sus puntos de referencia faciales (`landmarks`: ojos, nariz, boca, orejas) y la probabilidad de cuatro emociones — alegría, tristeza, ira y sorpresa.

Todos estos attributi usan la misma **enumeración de verosimilitud (`Likelihood`)**, con seis livelli: `UNKNOWN` (0), `VERY_UNLIKELY` (1), `UNLIKELY` (2), `POSSIBLE` (3), `LIKELY` (4) y `VERY_LIKELY` (5). La volverás a ver en Safe Search (lección 6).

**Una limitación importante**: la precisión depende de que el volto esté completamente visible — un volto parcialmente oculto puede no ser rilevato.

In [ ]:
LIKELIHOOD_NAMES = (
    "UNKNOWN",
    "VERY_UNLIKELY",
    "UNLIKELY",
    "POSSIBLE",
    "LIKELY",
    "VERY_LIKELY",
)

def detect_faces(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.face_detection(image=image)

    for face in response.face_annotations:
        print("Alegría:", LIKELIHOOD_NAMES[face.joy_likelihood])
        print("Ira:", LIKELIHOOD_NAMES[face.anger_likelihood])
        print("Sorpresa:", LIKELIHOOD_NAMES[face.surprise_likelihood])
        print("Tristeza:", LIKELIHOOD_NAMES[face.sorrow_likelihood])
        print("Confianza de detección:", face.detection_confidence)
        print("---")

    return response.face_annotations


# detect_faces("grupo.jpg")


## Ejercicio 2 — Emoción dominante por volto

Completa `emocion_dominante` para que, por cada volto rilevato, determine cuál de las cuatro emociones (alegría, ira, sorpresa, tristeza) tiene el valor de verosimilitud más alto, y devuelva una lista de tuplas `(indice_volto, emocion_dominante)`.

Pista: construye un diccionario `{"alegria": face.joy_likelihood, ...}` y usa `max(..., key=...)`.

In [ ]:
def emocion_dominante(image_path):
    faces = detect_faces(image_path)
    resultados = []

    for i, face in enumerate(faces):
        # TODO 1: construye el diccionario de emociones y sus likelihoods
        emociones = ...

        # TODO 2: obtén la emoción con el valor más alto
        dominante = ...

        resultados.append((i, dominante))

    return resultados


# print(emocion_dominante("grupo.jpg"))


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
def emocion_dominante(image_path):
    faces = detect_faces(image_path)
    risultati = []

    for i, face in enumerate(faces):
        emociones = {
            "alegria": face.joy_likelihood,
            "ira": face.anger_likelihood,
            "sorpresa": face.surprise_likelihood,
            "tristeza": face.sorrow_likelihood,
        }
        dominante = max(emociones, key=emociones.get)
        risultati.append((i, dominante))

    return risultati


print(emocion_dominante("grupo.jpg"))
```

</details>

## 3. Propiedades de la immagine (`image_properties`)

`image_properties` extrae los colorei dominprima di una immagine, cada uno con su valor RGB (rojo, verde y azul) y una punteggio de prevalencia (`score`) y de proporción de pixel (`pixel_fraction`).

**Ten en cuenta**: los colorei dominantes reflejan la prevalencia global de colorei en la immagine, no necesariamente el elemento visualmente más importante de la escena.

In [ ]:
def get_dominant_colors(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.image_properties(image=image)
    colors = response.image_properties_annotation.dominant_colors.colors

    dominant_colors = []
    for color_info in colors:
        rgb = (color_info.color.red, color_info.color.green, color_info.color.blue)
        dominant_colors.append(
            {"rgb": rgb, "score": color_info.score, "pixel_fraction": color_info.pixel_fraction}
        )

    return dominant_colors


# get_dominant_colors("atardecer.jpg")


## Ejercicio 3 — Paleta de colorei con Matplotlib

Completa `mostrar_paleta` para que dibuje un gráfico de barras horizontal con `matplotlib`, donde cada barra representa un colore dominante: el colore de relleno debe ser el RGB real, y la longitud de la barra debe representar el `pixel_fraction`.

In [ ]:
import matplotlib.pyplot as plt

def mostrar_paleta(image_path):
    colores = get_dominant_colors(image_path)

    fig, ax = plt.subplots(figsize=(6, 4))

    for i, info in enumerate(colores):
        # TODO 1: normaliza el RGB a valores entre 0 y 1 para matplotlib
        color_normalizado = ...

        # TODO 2: dibuja una barra horizontal (ax.barh) con longitud info["pixel_fraction"]
        ...

    ax.set_xlabel("Proporción de píxeles")
    ax.set_title("Colores dominantes")
    plt.show()


# mostrar_paleta("atardecer.jpg")


<details>
<summary><strong>Ver soluzione</strong></summary>

```Python
import matplotlib.pyplot as plt

def mostrar_paleta(image_path):
    colorei = get_dominant_colores(image_path)

    fig, ax = plt.subplots(figsize=(6, 4))

    for i, info in enumerate(colorei):
        r, g, b = info["rgb"]
        colore_normalizado = (r / 255, g / 255, b / 255)
        ax.barh(i, info["pixel_fraction"], colore=colore_normalizado)

    ax.set_xlabel("Proporción de pixel")
    ax.set_title("Colores dominantes")
    plt.show()


mostrar_paleta("atardecer.jpg")
```

</details>

---

**Siguiente paso:** abre `05_Etichettatura_de_Immagini.ipynb`.